# TD 学习（TD(0), TD(λ)）与 GAE 笔记（速查版）

> 关键词：**bootstrap（自举）**、**bias-variance（偏差-方差权衡）**、**n-step return**、**eligibility trace（资格迹）**、**GAE（Generalized Advantage Estimation）**  
> 适用：PPO/A2C/SAC 等需要优势估计（advantage estimation）的策略梯度/Actor-Critic 管线

---

## 1. TD(0)：一步自举更新

### 1.1 TD 误差（TD error）
对一次转移 \((s_t, a_t, r_{t+1}, s_{t+1})\)：
\[
\delta_t = r_{t+1} + \gamma V(s_{t+1}) - V(s_t)
\]

### 1.2 价值函数更新（state-value）
\[
V(s_t) \leftarrow V(s_t) + \alpha \, \delta_t
\]

- \(\alpha\)：学习率  
- \(\gamma\)：折扣因子（discount factor）

### 1.3 为什么叫 bootstrap（自举）？
**用当前已有的估计 \(V(s_{t+1})\) 构造目标来更新 \(V(s_t)\)**，无需等整个 episode 结束拿到完整回报。

---

## 2. TD vs Monte Carlo（MC）：核心差异（背诵版）

- **MC**：用完整回报 \(G_t\) 更新，**要等 episode 结束**，**无偏（unbiased）但高方差（high variance）**  
- **TD(0)**：用 \(r_{t+1}+\gamma V(s_{t+1})\) 更新，**不必等结束**，**有偏（biased）但低方差（low variance）**

---

## 3. TD(λ)：在 TD(0) 与 MC 之间插值

TD(λ) 用 \(\lambda \in [0,1]\) 控制“看多远”的程度，本质上是在 **n-step returns** 之间做加权平均。

### 3.1 Forward view（理论视角）：λ-return
定义 n-step return：
\[
G_t^{(n)} = \sum_{k=1}^{n} \gamma^{k-1} r_{t+k} + \gamma^n V(s_{t+n})
\]
λ-return（加权平均）：
\[
G_t^\lambda = (1-\lambda)\sum_{n=1}^{\infty}\lambda^{n-1} G_t^{(n)}
\]

- \(\lambda = 0\) → 只用 1-step → **TD(0)**  
- \(\lambda \to 1\) → 更偏长 horizon → 更像 **MC**（终止任务中可趋近 MC）

### 3.2 Backward view（工程视角）：Eligibility Trace（资格迹）
直觉：维护“最近访问过的状态/参数”的痕迹强度，把当前 TD 误差 \(\delta_t\) **按痕迹衰减**分摊给过去，实现在线更新（不用等未来很多步）。

---

## 4. GAE：优势估计的 TD(λ) 化（PPO/A2C 标配）

### 4.1 定义 TD residual（同 TD error）
\[
\delta_t = r_{t+1} + \gamma V(s_{t+1}) - V(s_t)
\]

### 4.2 GAE 的递推（倒序实现，最常用）
带终止截断（done mask）：
\[
A_t = \delta_t + \gamma\lambda (1-d_t)\,A_{t+1}
\]

- \(d_t\) 是 done 标记（终止时截断）
- 若 done=True（\(d_t=1\)），则：
\[
A_t = \delta_t
\]

最简版（不考虑 done）：
\[
A_t = \delta_t + \gamma\lambda A_{t+1}
\]

### 4.3 λ 的退化情况（务必记住）
- \(\lambda=0\)：  
\[
A_t = \delta_t
\]
即 1-step TD advantage（偏差大、方差小）
- \(\lambda\to1\)（终止任务）：更像 MC 优势（偏差小、方差大）

### 4.4 为什么 TD error 和 GAE 里都出现 \(\gamma\)？
是同一个 \(\gamma\)，含义一致：**折扣未来**  
- 在 \(\delta_t\) 里：折扣下一状态价值 \(V(s_{t+1})\)（Bellman 折扣）  
- 在 GAE 递推里：折扣“未来残差/优势回传”的影响（通过 \(\gamma\lambda\) 控制衰减）

---

## 5. 代码实现要点（伪代码）

### 5.1 计算 GAE（倒序）
```python
# inputs: rewards[0:T], values[0:T+1], dones[0:T]
# output: advantages[0:T]
gae = 0.0
for t in reversed(range(T)):
    delta = rewards[t] + gamma * values[t+1] * (1 - dones[t]) - values[t]
    gae = delta + gamma * lam * (1 - dones[t]) * gae
    advantages[t] = gae
```

### 5.2 由 advantage 得到 value target（用于 critic 回归）
常见做法：
\[
\hat V_t^{target} = A_t + V(s_t)
\]

---

## 6. 常见坑（你现在最容易踩的点）

1. **忘写 \(\gamma\)**：\(\delta_t = r + \gamma V_{t+1} - V_t\) 必须带 \(\gamma\)  
2. **done 没截断**：终止时要乘 \((1-d_t)\)，不然优势会“跨 episode 泄漏”  
3. **GAE ≠ Q-V 的单步差**：GAE 是“多步残差加权和”，不是只算一次 \(\delta_t\)  
4. **\(\lambda\) 不是“是否看重未来奖励”的单纯权重**：更准确是“n-step 回报参与程度/长度分布”的控制旋钮  
5. **数值稳定性**：PPO 常对 advantage 做标准化（mean=0, std=1）以稳定训练

---

## 7. 记忆卡片（只背这 5 行就够用）

- TD error：\(\delta_t = r_{t+1} + \gamma V_{t+1} - V_t\)  
- TD(0) 更新：\(V_t \leftarrow V_t + \alpha \delta_t\)  
- GAE 递推：\(A_t = \delta_t + \gamma\lambda(1-d_t)A_{t+1}\)  
- \(\lambda=0\) ⇒ \(A_t=\delta_t\)；\(\lambda\to1\) ⇒ 更像 MC  
- done=True ⇒ 截断：\(A_t=\delta_t\)

---

*生成时间：2026-01-13 02:16:17*